# 异步HogWild推理

一句话总结：
对于推理模型，并发N个，然后共享KV缓存，能够涌现协作能力。

In [ ]:

from dataclasses import dataclass, field
from typing import List, Literal
import random

Category = Literal[
    "A", "B", "noise", "coord"
]

@dataclass
class SharedCache:
    tokens: List[tuple[int, Category]] = field(default_factory=list)

    def counts() ->dict:
        c = {
            "A": 0,
            "B": 0,
            "noise": 0,
            "coord": 0,
        }
        for _, c in self.tokens:
            c[c] += 1
        return c

@dataclass
class Worker:
    id: int
    intended: Category
    coordinate_weight: float
    rng: random.Random

def decide_next_category(
    worker: Worker,
    cache: SharedCache,
    target_per_category: int
) -> Category:
    if worker.rng.random() < 0.05:
        return "noise"

    counts = cache.counts()
    base = worker.intended

    if worker.rng.random() < worker.coordinate_weight:
        candidates = sorted(("A", "B"), key=lambda c: counts[c])
        return candidates[0]

    if worker.rng.random() < 0.1:
        return "coord"

    return base

def run_hogwild(
    n_workers: int,
    step_budget: int,
    target_per_category: int,
    seed: int=42    
) -> dict:
    cache = SharedCache()
    workers = []
    for i in range(n_workers):
        workers.append(Worker(
            id = i,
            intended = "A",
            coordinate_weight = 0.0,
            rng = random.Random(seed + i)
        ))

    trace = List[tuple[int, Category, str]] = []
    step = 0
    progress = 0
    while step < step_budget:
        this_step_categories: List[tuple[int, Category]] = []
        for worker in workers:
            next_category = decide_next_category(
                worker,
                cache,
                target_per_category
            )
            cache.tokens.append(worker.id, next_category)
            this_step_categories.append((worker.id, next_category))

        seen_work_categories = set()
        for worker_id, category in this_step_categories:
            tag = "redundant"
            if category in ("A", "B") and category not in seen_work_categories:
                seen_work_categories.add(category)
                progress += 1
                tag = "unique"
            trace.append((step, category, tag))